# Lab 01 — Ingestão incremental e idempotente (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite).

Objetivo: sentir a **marca d'água** (incremental) e o **upsert idempotente** (rodar 2x sem duplicar).

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
# FONTE com updated_at
con.execute('CREATE TABLE fonte(id INTEGER, nome VARCHAR, updated_at DATE)')
con.executemany('INSERT INTO fonte VALUES (?,?,?)', [
    (1,'ana','2026-08-10'),(2,'bruno','2026-08-15'),(3,'caio','2026-08-20')])
# DESTINO já tem a ana (carga anterior)
con.execute('CREATE TABLE destino(id INTEGER PRIMARY KEY, nome VARCHAR)')
con.execute("INSERT INTO destino VALUES (1,'ana')")
con.execute('SELECT * FROM fonte ORDER BY id').df()

## 1. Incremental: só o que mudou desde a marca d'água
A última carga foi em `2026-08-12`. Trazemos só `updated_at > marca`.

In [ ]:
marca = '2026-08-12'
con.execute(f"SELECT id, nome FROM fonte WHERE updated_at > DATE '{marca}' ORDER BY id").df()

## 2. Upsert idempotente (ON CONFLICT)
Insere se novo, atualiza se existe. Rodar 2x **não** duplica.

In [ ]:
upsert = '''
  INSERT INTO destino (id, nome)
  SELECT id, nome FROM fonte WHERE updated_at > DATE '2026-08-12'
  ON CONFLICT (id) DO UPDATE SET nome = EXCLUDED.nome
'''
con.execute(upsert)   # 1a execução
con.execute(upsert)   # 2a execução (idempotência!)
con.execute('SELECT * FROM destino ORDER BY id').df()

## 3. Sua vez (mini-desafio)
Se a marca d'água fosse `2026-08-16`, **quais ids** seriam ingeridos? Traga a lista de `id` (crescente). Verifique.

In [ ]:
resposta = [r[0] for r in con.execute("SELECT id FROM fonte WHERE updated_at > DATE '2026-08-16' ORDER BY id").fetchall()]
resposta

In [ ]:
def verificar(ids):
    try:
        assert ids == [3], 'Só caio (2026-08-20) é > 2026-08-16.'
        print('✅ Correto! Marca dágua = trazer só o que mudou depois dela.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)